# O1-05 — First Gamma Compatibility Detector

This notebook trains the first pre-inference compatibility detector for Gemfinder.

The detector uses only features available before neural-network inference.

Training labels come from the high-confidence cases generated in O1-04:

- Compatible
- Incompatible

Ambiguous cases are excluded from the first classifier experiment.

The main evaluation priority is recall of the incompatible class because a false
negative would allow an unsuitable instance to continue to the existing Gamma
neural-network pathway.

In [6]:
from pathlib import Path

import numpy as np
import pandas as pd

BASE_DIR = Path(r"C:\AIML Y2 S2\Capstone 1\RYA")

FEATURE_FILE = (
    BASE_DIR / "O1_classifier_inputs_unlabelled.csv"
)

LABEL_FILE = (
    BASE_DIR / "O1_gamma_fit_pilot_zones_v1.csv"
)

print("Feature file exists:", FEATURE_FILE.exists())
print("Label file exists:", LABEL_FILE.exists())

Feature file exists: True
Label file exists: True


## 1. Load screening features and high-confidence compatibility labels

Load the complete O1 screening-feature dataset and the compatibility labels
generated by the O1-04 Gamma-fit pilot.

In [9]:
features = pd.read_csv(FEATURE_FILE)
labels = pd.read_csv(LABEL_FILE)

print("Feature dataset:", features.shape)
print("Label dataset:", labels.shape)

print("\nCompatibility zones:")
print(
    labels["compatibility_zone_v1"]
    .value_counts()
)

Feature dataset: (63328, 57)
Label dataset: (175, 43)

Compatibility zones:
compatibility_zone_v1
compatible      89
ambiguous       51
incompatible    35
Name: count, dtype: int64


## 2. Build the high-confidence classifier dataset

Only high-confidence compatibility labels are used for the first detector.

The 51 ambiguous pilot cases are excluded from this initial experiment.

The remaining compatible and incompatible labels are merged back to the 54
pre-inference screening features using `exp_code`.

In [12]:
high_conf_labels = labels[
    labels["compatibility_zone_v1"].isin(
        ["compatible", "incompatible"]
    )
][
    ["exp_code", "compatibility_zone_v1"]
].copy()

print("High-confidence labels:", high_conf_labels.shape)

print(
    high_conf_labels["compatibility_zone_v1"]
    .value_counts()
)

High-confidence labels: (124, 2)
compatibility_zone_v1
compatible      89
incompatible    35
Name: count, dtype: int64


In [14]:
detector_df = features.merge(
    high_conf_labels,
    on="exp_code",
    how="inner",
    validate="one_to_one"
)

print("Merged detector dataset:", detector_df.shape)

print("\nLabel counts:")
print(
    detector_df["compatibility_zone_v1"]
    .value_counts()
)

print("\nMissing values after merge:")
print(detector_df.isna().sum().sum())

Merged detector dataset: (124, 58)

Label counts:
compatibility_zone_v1
compatible      89
incompatible    35
Name: count, dtype: int64

Missing values after merge:
0


## 3. Define classifier inputs and check for train-test leakage

The detector target is binary:

- `0` = compatible
- `1` = incompatible

Only the 54 pre-inference screening features are used as model inputs.

Identifier columns are excluded from training.

Before splitting the data, mirrored X/Y instances are checked because the
Gemfinder distance is symmetric with respect to exchanging the two respondent
groups. Mirrored copies must not be placed across training and test sets.

In [17]:
identifier_cols = [
    "exp_code",
    "P_matrix_question",
    "source_file"
]

target_col = "compatibility_zone_v1"

feature_cols = [
    col for col in detector_df.columns
    if col not in identifier_cols + [target_col]
]

X = detector_df[feature_cols].copy()

y = (
    detector_df[target_col]
    .map({
        "compatible": 0,
        "incompatible": 1
    })
    .astype(int)
)

print("X shape:", X.shape)
print("y shape:", y.shape)

print("\nTarget counts:")
print(y.value_counts().sort_index())

print("\nNumber of model features:", len(feature_cols))

print("\nMissing values in X:", X.isna().sum().sum())
print("Duplicated feature names:", X.columns.duplicated().sum())

X shape: (124, 54)
y shape: (124,)

Target counts:
compatibility_zone_v1
0    89
1    35
Name: count, dtype: int64

Number of model features: 54

Missing values in X: 0
Duplicated feature names: 0


In [19]:
def make_mirror_key(row, decimals=8):
    P = tuple(
        round(float(row[f"P_{i}{j}"]), decimals)
        for i in range(1, 5)
        for j in range(1, 5)
    )

    group_X = (
        int(row["n_X"]),
        tuple(
            round(float(row[f"X_{i}"]), decimals)
            for i in range(1, 5)
        )
    )

    group_Y = (
        int(row["n_Y"]),
        tuple(
            round(float(row[f"Y_{i}"]), decimals)
            for i in range(1, 5)
        )
    )

    # Canonical ordering makes X/Y swaps share the same key
    group_pair = tuple(
        sorted(
            [group_X, group_Y],
            key=str
        )
    )

    return (P, group_pair)


detector_df["mirror_group"] = detector_df.apply(
    make_mirror_key,
    axis=1
)

mirror_counts = (
    detector_df["mirror_group"]
    .value_counts()
)

print("Unique mirror groups:", mirror_counts.size)

print(
    "Mirror groups containing >1 row:",
    (mirror_counts > 1).sum()
)

print(
    "Rows belonging to repeated mirror groups:",
    mirror_counts[mirror_counts > 1].sum()
)

Unique mirror groups: 124
Mirror groups containing >1 row: 0
Rows belonging to repeated mirror groups: 0


## 4. Create a reproducible stratified train-test split

No mirrored X/Y instances were found among the 124 high-confidence pilot cases.

A stratified split is therefore used for the first baseline experiment so that
the compatible/incompatible class proportions are approximately preserved in
both training and test sets.

Because the labelled pilot is small, this hold-out result will later be
supplemented with repeated stratified cross-validation.

In [22]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.25,
    stratify=y,
    random_state=42
)

print("Training set:", X_train.shape)
print("Test set:", X_test.shape)

print("\nTraining target counts:")
print(y_train.value_counts().sort_index())

print("\nTest target counts:")
print(y_test.value_counts().sort_index())

Training set: (93, 54)
Test set: (31, 54)

Training target counts:
compatibility_zone_v1
0    67
1    26
Name: count, dtype: int64

Test target counts:
compatibility_zone_v1
0    22
1     9
Name: count, dtype: int64


## 5. Train first baseline compatibility classifiers

Three interpretable baseline models are trained:

- Logistic Regression
- Random Forest
- Gradient Boosting

The primary evaluation priority is recall of the incompatible class (`1`),
because a false negative would allow an unsuitable instance to continue to the
existing Gamma neural-network pathway.

Logistic Regression uses feature scaling because the screening features have
different numerical ranges.

In [25]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import (
    RandomForestClassifier,
    GradientBoostingClassifier
)

models = {
    "Logistic Regression": Pipeline([
        ("scaler", StandardScaler()),
        ("model", LogisticRegression(
            class_weight="balanced",
            max_iter=5000,
            random_state=42
        ))
    ]),

    "Random Forest": RandomForestClassifier(
        n_estimators=500,
        class_weight="balanced",
        random_state=42
    ),

    "Gradient Boosting": GradientBoostingClassifier(
        random_state=42
    )
}

print("Models:")
for name in models:
    print("-", name)

Models:
- Logistic Regression
- Random Forest
- Gradient Boosting


In [27]:
fitted_models = {}

for name, model in models.items():
    model.fit(X_train, y_train)
    fitted_models[name] = model
    print(f"Trained: {name}")

Trained: Logistic Regression
Trained: Random Forest
Trained: Gradient Boosting


## 6. Evaluate baseline classifiers

The first evaluation focuses on the incompatible class (`1`).

Key metrics are:

- recall of incompatible cases;
- precision;
- F1-score;
- confusion matrix;
- number of false negatives.

False negatives are particularly important because they represent incompatible
instances that would incorrectly be allowed to proceed to the existing Gamma
neural-network pathway.

In [30]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report
)

evaluation_rows = []

for name, model in fitted_models.items():

    y_pred = model.predict(X_test)

    cm = confusion_matrix(
        y_test,
        y_pred,
        labels=[0, 1]
    )

    tn, fp, fn, tp = cm.ravel()

    evaluation_rows.append({
        "model": name,
        "accuracy": accuracy_score(y_test, y_pred),
        "precision_incompatible": precision_score(
            y_test, y_pred, pos_label=1, zero_division=0
        ),
        "recall_incompatible": recall_score(
            y_test, y_pred, pos_label=1, zero_division=0
        ),
        "f1_incompatible": f1_score(
            y_test, y_pred, pos_label=1, zero_division=0
        ),
        "true_negatives": tn,
        "false_positives": fp,
        "false_negatives": fn,
        "true_positives": tp
    })

evaluation_df = pd.DataFrame(evaluation_rows)

evaluation_df.sort_values(
    "recall_incompatible",
    ascending=False
)

,model,accuracy,precision_incompatible,recall_incompatible,f1_incompatible,true_negatives,false_positives,false_negatives,true_positives
0,Logistic Regression,0.870968,0.777778,0.777778,0.777778,20,2,2,7
2,Gradient Boosting,0.870968,0.777778,0.777778,0.777778,20,2,2,7
1,Random Forest,0.870968,0.857143,0.666667,0.750000,21,1,3,6


In [32]:
for name, model in fitted_models.items():

    y_pred = model.predict(X_test)

    print("\n" + "=" * 60)
    print(name)
    print("=" * 60)

    print("\nConfusion matrix:")
    print(
        confusion_matrix(
            y_test,
            y_pred,
            labels=[0, 1]
        )
    )

    print("\nClassification report:")
    print(
        classification_report(
            y_test,
            y_pred,
            target_names=[
                "compatible",
                "incompatible"
            ],
            digits=3,
            zero_division=0
        )
    )


Logistic Regression

Confusion matrix:
[[20  2]
 [ 2  7]]

Classification report:
              precision    recall  f1-score   support

  compatible      0.909     0.909     0.909        22
incompatible      0.778     0.778     0.778         9

    accuracy                          0.871        31
   macro avg      0.843     0.843     0.843        31
weighted avg      0.871     0.871     0.871        31


Random Forest

Confusion matrix:
[[21  1]
 [ 3  6]]

Classification report:
              precision    recall  f1-score   support

  compatible      0.875     0.955     0.913        22
incompatible      0.857     0.667     0.750         9

    accuracy                          0.871        31
   macro avg      0.866     0.811     0.832        31
weighted avg      0.870     0.871     0.866        31


Gradient Boosting

Confusion matrix:
[[20  2]
 [ 2  7]]

Classification report:
              precision    recall  f1-score   support

  compatible      0.909     0.909     0.909       

## 7. Repeated stratified cross-validation

The hold-out test set contains only nine incompatible cases, so model performance
can change noticeably if only one or two cases move between splits.

Repeated stratified cross-validation is therefore used to estimate performance
more reliably across multiple train/test partitions.

The main metric remains recall of the incompatible class, with false negatives
treated as the most important error.

In [35]:
from sklearn.model_selection import RepeatedStratifiedKFold, cross_validate
from sklearn.metrics import make_scorer

recall_incompatible_scorer = make_scorer(
    recall_score,
    pos_label=1,
    zero_division=0
)

precision_incompatible_scorer = make_scorer(
    precision_score,
    pos_label=1,
    zero_division=0
)

f1_incompatible_scorer = make_scorer(
    f1_score,
    pos_label=1,
    zero_division=0
)

scoring = {
    "accuracy": "accuracy",
    "precision_incompatible": precision_incompatible_scorer,
    "recall_incompatible": recall_incompatible_scorer,
    "f1_incompatible": f1_incompatible_scorer
}

cv = RepeatedStratifiedKFold(
    n_splits=5,
    n_repeats=10,
    random_state=42
)

cv_results = []

for name, model in models.items():

    scores = cross_validate(
        model,
        X,
        y,
        cv=cv,
        scoring=scoring,
        n_jobs=-1
    )

    cv_results.append({
        "model": name,

        "accuracy_mean":
            scores["test_accuracy"].mean(),

        "accuracy_std":
            scores["test_accuracy"].std(),

        "precision_incompatible_mean":
            scores["test_precision_incompatible"].mean(),

        "recall_incompatible_mean":
            scores["test_recall_incompatible"].mean(),

        "recall_incompatible_std":
            scores["test_recall_incompatible"].std(),

        "f1_incompatible_mean":
            scores["test_f1_incompatible"].mean()
    })

cv_summary = pd.DataFrame(cv_results)

cv_summary.sort_values(
    "recall_incompatible_mean",
    ascending=False
)

,model,accuracy_mean,accuracy_std,precision_incompatible_mean,recall_incompatible_mean,recall_incompatible_std,f1_incompatible_mean
0,Logistic Regression,0.883933,0.067528,0.781129,0.851429,0.124409,0.807745
2,Gradient Boosting,0.890267,0.063305,0.808637,0.817143,0.173887,0.802276
1,Random Forest,0.907933,0.054236,0.912175,0.754286,0.176219,0.812275


## 8. Tune the decision threshold for incompatible-case recall

The default binary classification threshold is 0.50.

For the Gemfinder screening gate, false negatives are more costly than false
positives because a false negative allows an incompatible instance to continue
to the existing Gamma neural-network pathway.

The Logistic Regression threshold is therefore evaluated across a range of
values to examine the trade-off between incompatible-case recall and precision.

In [37]:
from sklearn.base import clone

cv_threshold = RepeatedStratifiedKFold(
    n_splits=5,
    n_repeats=10,
    random_state=42
)

probability_records = []

for fold_id, (train_idx, test_idx) in enumerate(
    cv_threshold.split(X, y),
    start=1
):
    model = clone(
        models["Logistic Regression"]
    )

    model.fit(
        X.iloc[train_idx],
        y.iloc[train_idx]
    )

    probs = model.predict_proba(
        X.iloc[test_idx]
    )[:, 1]

    for idx, prob in zip(test_idx, probs):
        probability_records.append({
            "row_index": idx,
            "true_label": int(y.iloc[idx]),
            "prob_incompatible": float(prob),
            "fold_id": fold_id
        })


repeated_oof = pd.DataFrame(
    probability_records
)

print("Prediction records:", repeated_oof.shape)

print("\nPredictions per original case:")
print(
    repeated_oof["row_index"]
    .value_counts()
    .describe()
)

print("\nProbability range:")
print(
    repeated_oof["prob_incompatible"]
    .describe()
)

Prediction records: (1240, 4)

Predictions per original case:
count    124.0
mean      10.0
std        0.0
min       10.0
25%       10.0
50%       10.0
75%       10.0
max       10.0
Name: count, dtype: float64

Probability range:
count    1.240000e+03
mean     3.198314e-01
std      3.977931e-01
min      7.794546e-07
25%      1.243149e-03
50%      3.083231e-02
75%      7.423631e-01
max      1.000000e+00
Name: prob_incompatible, dtype: float64


In [38]:
mean_oof = (
    repeated_oof
    .groupby(
        ["row_index", "true_label"],
        as_index=False
    )
    .agg(
        mean_prob_incompatible=(
            "prob_incompatible",
            "mean"
        ),
        std_prob_incompatible=(
            "prob_incompatible",
            "std"
        )
    )
)

print("Averaged cases:", mean_oof.shape)

print("\nMean probability summary:")
print(
    mean_oof["mean_prob_incompatible"]
    .describe()
)

print("\nProbability by true class:")
print(
    mean_oof.groupby("true_label")[
        "mean_prob_incompatible"
    ].describe()
)

Averaged cases: (124, 4)

Mean probability summary:
count    124.000000
mean       0.319831
std        0.389737
min        0.000002
25%        0.001661
50%        0.053513
75%        0.744121
max        1.000000
Name: mean_prob_incompatible, dtype: float64

Probability by true class:
            count      mean       std       min       25%       50%       75%  \
true_label                                                                      
0            89.0  0.122732  0.237688  0.000002  0.000836  0.005218  0.065861   
1            35.0  0.821026  0.207371  0.331547  0.708378  0.913260  0.986877   

                max  
true_label           
0           0.99505  
1           1.00000  


In [39]:
thresholds = [
    0.50,
    0.45,
    0.40,
    0.35,
    0.30,
    0.25,
    0.20,
    0.15,
    0.10
]

threshold_rows = []

for threshold in thresholds:

    y_true = mean_oof["true_label"]

    y_pred_threshold = (
        mean_oof["mean_prob_incompatible"]
        >= threshold
    ).astype(int)

    tn, fp, fn, tp = confusion_matrix(
        y_true,
        y_pred_threshold,
        labels=[0, 1]
    ).ravel()

    threshold_rows.append({
        "threshold": threshold,
        "recall_incompatible": recall_score(
            y_true,
            y_pred_threshold,
            pos_label=1,
            zero_division=0
        ),
        "precision_incompatible": precision_score(
            y_true,
            y_pred_threshold,
            pos_label=1,
            zero_division=0
        ),
        "f1_incompatible": f1_score(
            y_true,
            y_pred_threshold,
            pos_label=1,
            zero_division=0
        ),
        "false_negatives": fn,
        "false_positives": fp,
        "true_positives": tp,
        "true_negatives": tn
    })

threshold_df = pd.DataFrame(
    threshold_rows
)

threshold_df

,threshold,recall_incompatible,precision_incompatible,f1_incompatible,false_negatives,false_positives,true_positives,true_negatives
0,0.50,0.885714,0.794872,0.837838,4,8,31,81
1,0.45,0.914286,0.761905,0.831169,3,10,32,79
2,0.40,0.942857,0.717391,0.814815,2,13,33,76
3,0.35,0.942857,0.702128,0.804878,2,14,33,75
4,0.30,1.000000,0.700000,0.823529,0,15,35,74
5,0.25,1.000000,0.700000,0.823529,0,15,35,74
6,0.20,1.000000,0.673077,0.804598,0,17,35,72
7,0.15,1.000000,0.648148,0.786517,0,19,35,70
8,0.10,1.000000,0.614035,0.760870,0,22,35,67


## 9. Select the provisional high-recall operating threshold

Threshold sensitivity analysis shows that a probability threshold of 0.30
detects all high-confidence incompatible cases in the current repeated
out-of-fold pilot evaluation.

At threshold 0.30:

- incompatible recall = 1.00
- false negatives = 0
- false positives = 15
- incompatible precision = 0.70

Lowering the threshold below 0.30 does not improve recall further and increases
the number of compatible cases unnecessarily intercepted.

Therefore, 0.30 is selected as the provisional v1 screening threshold.

This threshold is not treated as a final production setting and requires
validation on a larger labelled simulation dataset.

In [45]:
PROVISIONAL_THRESHOLD = 0.30

print("Provisional v1 threshold:", PROVISIONAL_THRESHOLD)

Provisional v1 threshold: 0.3


## 10. Inspect false positives at the provisional threshold

At the provisional threshold of 0.30, all high-confidence incompatible pilot
cases are detected, but 15 compatible cases are also intercepted.

These false positives are inspected to determine whether they share input
characteristics associated with higher compatibility risk, such as extreme
group-size imbalance, sparse histograms, degeneracy or unusual matrix structure.

In [48]:
mean_oof_with_info = mean_oof.merge(
    detector_df[
        [
            "exp_code",
            "P_matrix_question",
            "source_file",
            "group_size_ratio",
            "active_categories_X",
            "active_categories_Y",
            "degenerate_X",
            "degenerate_Y",
            "sparse_count_X",
            "sparse_count_Y",
            "asymmetry_P",
            "all_rows_diagonal_mode_P"
        ]
    ].reset_index().rename(
        columns={"index": "row_index"}
    ),
    on="row_index",
    how="left",
    validate="one_to_one"
)

mean_oof_with_info["pred_v1"] = (
    mean_oof_with_info["mean_prob_incompatible"]
    >= PROVISIONAL_THRESHOLD
).astype(int)

false_positives = mean_oof_with_info[
    (mean_oof_with_info["true_label"] == 0)
    & (mean_oof_with_info["pred_v1"] == 1)
].copy()

print("False positives:", len(false_positives))

false_positives[
    [
        "exp_code",
        "mean_prob_incompatible",
        "std_prob_incompatible",
        "group_size_ratio",
        "active_categories_X",
        "active_categories_Y",
        "degenerate_X",
        "degenerate_Y",
        "sparse_count_X",
        "sparse_count_Y",
        "asymmetry_P",
        "all_rows_diagonal_mode_P"
    ]
].sort_values(
    "mean_prob_incompatible",
    ascending=False
)

False positives: 15


,exp_code,mean_prob_incompatible,std_prob_incompatible,group_size_ratio,active_categories_X,active_categories_Y,degenerate_X,degenerate_Y,sparse_count_X,sparse_count_Y,asymmetry_P,all_rows_diagonal_mode_P
90,orthog_B_1_12277,0.995050,0.002954,9.200000,1,4,1,0,3,0,0.009418,1
89,orthog_B_1_11857,0.805829,0.130862,51.100000,4,2,0,0,0,2,0.005098,1
38,orthog_B_4_14271,0.805376,0.089234,1.307692,3,1,0,1,1,3,0.003361,1
75,orthog_B_1_6718,0.780026,0.126435,108.666667,4,3,0,0,0,1,0.006471,1
61,orthog_B_4_29608,0.751445,0.126347,52.833333,4,2,0,0,0,2,0.022287,1
0,orthog_B_3_large_classes_134,0.741679,0.167384,120.361111,3,4,0,0,1,0,0.010321,1
4,orthog_B_3_large_classes_2004,0.732374,0.123795,112.750000,4,4,0,0,0,0,0.006866,1
51,orthog_B_4_23234,0.515877,0.226086,63.500000,4,4,0,0,0,0,0.004856,1
13,orthog_B_4_4345,0.492843,0.138895,111.333333,4,3,0,0,0,1,0.017456,1
93,orthog_B_1_13714,0.490703,0.253101,21.250000,4,1,0,1,0,3,0.016716,1


## 11. Save O1-05 baseline detector results

The first compatibility detector was evaluated using repeated stratified
cross-validation and threshold sensitivity analysis.

Logistic Regression achieved the highest mean incompatible-case recall among
the three baseline models and was selected for provisional threshold tuning.

A threshold of 0.30 achieved zero false negatives on the current 124
high-confidence pilot cases under repeated out-of-fold evaluation, while
producing 15 false positives.

The detector remains provisional and requires validation on a larger labelled
simulation dataset.

In [51]:
CV_OUTPUT = (
    BASE_DIR / "O1_05_cross_validation_summary.csv"
)

THRESHOLD_OUTPUT = (
    BASE_DIR / "O1_05_threshold_analysis.csv"
)

FALSE_POSITIVE_OUTPUT = (
    BASE_DIR / "O1_05_false_positives_v1.csv"
)

cv_summary.to_csv(
    CV_OUTPUT,
    index=False
)

threshold_df.to_csv(
    THRESHOLD_OUTPUT,
    index=False
)

false_positives.to_csv(
    FALSE_POSITIVE_OUTPUT,
    index=False
)

print("Saved:")
print(CV_OUTPUT)
print(THRESHOLD_OUTPUT)
print(FALSE_POSITIVE_OUTPUT)

Saved:
C:\AIML Y2 S2\Capstone 1\RYA\O1_05_cross_validation_summary.csv
C:\AIML Y2 S2\Capstone 1\RYA\O1_05_threshold_analysis.csv
C:\AIML Y2 S2\Capstone 1\RYA\O1_05_false_positives_v1.csv
